In [1]:
import pandas as pd
import re
import tools

In [2]:
df_kalendarz = pd.read_parquet("dane/interim/kalendarz_pelny_towid.parquet")

### Towary wazone 

In [3]:
# ============================================================
# SOFT FILTER — Kryterium 2: Towary ważone (per TowId)
# Krok 1: % transakcji z ułamkową ilością per TowId
# ============================================================

wazone_check = df_kalendarz.groupby('TowId').agg(
    LiczbaTransakcji=('IloscPlus', 'count'),
    LiczbaUlamkowych=('IloscPlus', lambda x: (x % 1 != 0).sum()),
).reset_index()

wazone_check['PctUlamkowych'] = (
    wazone_check['LiczbaUlamkowych'] / wazone_check['LiczbaTransakcji'] * 100
)

print(wazone_check['PctUlamkowych'].describe())

count     12481.000000
mean      10986.080719
std       23723.587967
min           0.745558
25%         380.416667
50%        1795.000000
75%        9291.666667
max      112700.000000
Name: PctUlamkowych, dtype: float64


In [4]:


def czy_nazwa_sugeruje_wage(nazwa):
    nazwa = str(nazwa).upper().strip()
    
    # Wyklucz przypadki "liczba+KG" (stała gramatura opakowania, np. "0,5KG", "1,65KG")
    bez_gramatury = re.sub(r'\d+[\.,]?\d*\s*KG\b', '', nazwa)
    
    # Szukamy samodzielnego "KG" jako jednostki sprzedaży (bez liczby bezpośrednio przed nim)
    return bool(re.search(r'\bKG\b', bez_gramatury))

# Szybki test kontrolny
testy = [
    "Chleb mieszany 0.6 kg PRECELEK",         # False - gramatura
    "PROSZEK PERSIL UNIVERSAL 1,65KG HENKEL", # False - gramatura
    "MĄKA ZIEMNIACZANA 0,5KG",                # False - gramatura
    "SURÓWKA KG",                              # True - waga
    "BAKŁAŻAN KG POLSKA",                      # True - waga
    "SAŁATKA WIELKANOCNA KG GRZEŚKOWIAK",      # True - waga
]
for t in testy:
    print(f"{czy_nazwa_sugeruje_wage(t)}: {t}")

False: Chleb mieszany 0.6 kg PRECELEK
False: PROSZEK PERSIL UNIVERSAL 1,65KG HENKEL
False: MĄKA ZIEMNIACZANA 0,5KG
True: SURÓWKA KG
True: BAKŁAŻAN KG POLSKA
True: SAŁATKA WIELKANOCNA KG GRZEŚKOWIAK


In [5]:
nazwy_do_sprawdzenia = df_kalendarz[['TowId', 'NazwaTow']].drop_duplicates(subset='TowId')
nazwy_do_sprawdzenia['NazwaSugerujeWage'] = nazwy_do_sprawdzenia['NazwaTow'].apply(czy_nazwa_sugeruje_wage)

wazone_check_pelne = wazone_check.merge(nazwy_do_sprawdzenia, on='TowId', how='left')

wazone_check_pelne['JestWazony'] = (
    (wazone_check_pelne['PctUlamkowych'] >= 50) |
    (wazone_check_pelne['NazwaSugerujeWage'])
)

lista_wazonych_towid = set(wazone_check_pelne[wazone_check_pelne['JestWazony']]['TowId'])
print(f"Ważonych TowId (finalne kryterium): {len(lista_wazonych_towid)}")

dodane_przez_nazwe = wazone_check_pelne[
    (wazone_check_pelne['NazwaSugerujeWage']) & 
    (wazone_check_pelne['PctUlamkowych'] < 50)
]
print(f"\nDodatkowo złapane przez samodzielne 'KG' (bez liczby): {len(dodane_przez_nazwe)}")
print(dodane_przez_nazwe[['TowId', 'NazwaTow', 'PctUlamkowych']].sort_values('NazwaTow'))

Ważonych TowId (finalne kryterium): 11935

Dodatkowo złapane przez samodzielne 'KG' (bez liczby): 0
Empty DataFrame
Columns: [TowId, NazwaTow, PctUlamkowych]
Index: []


In [6]:
df_kalendarz['JestWazony'] = df_kalendarz['TowId'].isin(lista_wazonych_towid)

print(df_kalendarz['JestWazony'].value_counts())

nazwy_wazonych = (
    df_kalendarz[df_kalendarz['TowId'].isin(lista_wazonych_towid)]
    [['TowId', 'NazwaTow', 'NazwaAsort']]
    .drop_duplicates(subset='TowId')
)
print(nazwy_wazonych['NazwaAsort'].value_counts())

JestWazony
True     14315234
False     1494927
Name: count, dtype: int64
NazwaAsort
***PRZECENY                         887
MARKA WŁASNA SPAR                   551
3 WÓDKI I NAP.ALK POWYŻEJ 18%       452
LODY                                373
CIASTKA                             351
                                   ... 
ZESTAWY I KOSMETYCZKI /kosmetyki      1
ZABAWKI /przemysłowe                  1
LEKI /farmaceutyki                    1
PRZETWORY DANIA GOTOWE                1
x PAPIERNICZO HIGIENICZNE             1
Name: count, Length: 238, dtype: int64


In [7]:
df_kalendarz['JestWazony'] = df_kalendarz['TowId'].isin(lista_wazonych_towid)

print(df_kalendarz['JestWazony'].value_counts())

nazwy_wazonych = (
    df_kalendarz[df_kalendarz['TowId'].isin(lista_wazonych_towid)]
    [['TowId', 'NazwaTow', 'NazwaAsort']]
    .drop_duplicates(subset='TowId')
)
print(nazwy_wazonych['NazwaAsort'].value_counts())

JestWazony
True     14315234
False     1494927
Name: count, dtype: int64
NazwaAsort
***PRZECENY                         887
MARKA WŁASNA SPAR                   551
3 WÓDKI I NAP.ALK POWYŻEJ 18%       452
LODY                                373
CIASTKA                             351
                                   ... 
ZESTAWY I KOSMETYCZKI /kosmetyki      1
ZABAWKI /przemysłowe                  1
LEKI /farmaceutyki                    1
PRZETWORY DANIA GOTOWE                1
x PAPIERNICZO HIGIENICZNE             1
Name: count, Length: 238, dtype: int64


In [8]:
df_kalendarz.to_parquet(
    "dane/interim/fact_inka_hard_flagged_wazone.parquet",
    compression='zstd',
    index=False
)
print(f"Zapisano: {df_kalendarz.shape}")

Zapisano: (15810161, 34)


In [ ]:
# suma kontrolna
nazwa_pliku = "fact_inka_hard_flagged_wazone.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")
#Mój hash (posortowane):   fact_inka_hard_flagged_wazone.parquet   a6de55437d9114c6b217a74e5fc7a773c5d21f882c754ab8c77453bbed8968f8

Mój hash (posortowane):   fact_inka_hard_flagged_wazone.parquet   a6de55437d9114c6b217a74e5fc7a773c5d21f882c754ab8c77453bbed8968f8
